In [14]:
#!/usr/bin/env python3
"""
Fit DDM and SRDM to rr98 WITHOUT a lapse mixture.

Instead of a lapse process, trials are trimmed (fastest/slowest 1% per
participant x instruction x difficulty cell removed, Ratcliff 2008 style)
and t0 is hard-bounded above by the fastest surviving RT for that
participant (t0_hi = min(RT) over the trimmed data). No blend, no
p_lapse, no mixture -- the model likelihood is evaluated directly.

Uses physical-brightness correctness (strength > 16 = bright).
Excludes strength == 16 (exactly ambiguous), same as the lapse version.

OUTPUT FILES (one per model, all participants stacked, same convention
as the lapse-model fitting cell):
  - fits_ddm_nolapse.csv
  - fits_srdm_nolapse.csv
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from datetime import datetime, timezone

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

# ╔═══════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — edit these                              ║
# ╚═══════════════════════════════════════════════════════════╝
for k in [16]:
    N_LEVELS      = k      # <-- CHANGE THIS: 7, 11, 16, 33, etc. (16 was the
                            #     BIC-best resolution found with the lapse models)
    DATA_PATH     = "../../rr98.csv"
    STAN_DDM      = "DDM_rr98_nolapse_native.stan"
    STAN_SRDM     = "SRDM_rr98_c_only_nolapse.stan"
    PARTICIPANTS  = ["jf", "kr", "nh"]
    TRIM_LOW      = 0.01    # trim fastest 1%
    TRIM_HIGH     = 0.99    # trim slowest 1% (i.e. keep [1st, 99th] percentile)

    DDM_OUT  = "fits_ddm_nolapse.csv"
    SRDM_OUT = "fits_srdm_nolapse.csv"


    # ═══════════════════════════════════════════════════════════
    # Data prep
    # ═══════════════════════════════════════════════════════════
    def load_data():
        df = pd.read_csv(DATA_PATH)
        df = df[df["outlier"] == False].copy()
        df["correct"] = df["correct"].astype(int)
        df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})

        # Physical correctness
        df = df[df["strength"] != 16].copy()
        df["act_correct"] = ((df["strength"] > 16) == (df["response"] == "light")).astype(int)

        # Bin strength into N_LEVELS groups (identical logic to the lapse version)
        if N_LEVELS == 33:
            unique_strengths = sorted(df["strength"].unique())
            strength_to_level = {s: i+1 for i, s in enumerate(unique_strengths)}
            df["diff_level"] = df["strength"].map(strength_to_level)
            actual_levels = len(unique_strengths)
        else:
            def _qcut_levels(s):
                return pd.qcut(s, q=N_LEVELS, labels=False, duplicates="drop") + 1
            df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
            actual_levels = df["diff_level"].nunique()

        df["cell"] = (df["sat_id"] - 1) * actual_levels + df["diff_level"]

        print(f"N_LEVELS requested: {N_LEVELS}, actual unique levels: {actual_levels}")
        print(f"Trials before trimming: {len(df)}")

        return df, actual_levels


    def trim_extremes(df, low=TRIM_LOW, high=TRIM_HIGH):
        """Trim the fastest/slowest tails within each (id, cell) group, i.e.
        per participant x instruction x difficulty cell -- not globally."""
        def _trim_group(g):
            lo, hi = g["rt"].quantile([low, high])
            return g[(g["rt"] >= lo) & (g["rt"] <= hi)]

        trimmed = df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)
        print(f"Trials after trimming ({low:.0%}/{high:.0%} per id x cell): "
            f"{len(trimmed)}  (dropped {len(df) - len(trimmed)}, "
            f"{(1 - len(trimmed)/len(df)):.1%})")
        return trimmed


    def build_data(df, pid, n_levels):
        d = df[df["id"] == pid]
        d_correct = d[d["act_correct"] == 1]
        d_false = d[d["act_correct"] == 0]
        t0_hi = float(d["rt"].min())   # hard bound: fastest TRIMMED RT for this pid
        return {
            "N_LEVELS": n_levels,
            "N_correct": len(d_correct), "N_false": len(d_false),
            "rt_correct": d_correct["rt"].to_numpy(),
            "rt_false": d_false["rt"].to_numpy(),
            "cell_correct": d_correct["cell"].to_numpy(dtype=int),
            "cell_false": d_false["cell"].to_numpy(dtype=int),
            "t0_hi": t0_hi,
        }


    # ═══════════════════════════════════════════════════════════
    # Fitting
    # ═══════════════════════════════════════════════════════════
    def fit_ddm(model, data):
        nl = data["N_LEVELS"]
        inits = {
            "a": [0.8, 1.5], "v_base": [2.0]*nl,
            "sv": 0.5, "sz": 0.1,
            "t0": 0.2 * data["t0_hi"],
        }
        return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                            iter=5000, show_console=True)


    def fit_srdm(model, data):
        nl = data["N_LEVELS"]
        inits = {
            "c": [0.0, 0.0], "B": 1.0,
            "t0": 0.5 * data["t0_hi"],
            "d_base": [1.5]*nl, "r": 4.5,
        }
        return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                            iter=5000, show_console=True)


    def aic_bic(mle, n_params):
        p = mle.optimized_params_pd
        ll_cols = [c for c in p.columns if c.startswith("log_lik")]
        total_ll = p[ll_cols].iloc[0].sum()
        n = len(ll_cols)
        return {"n_params": n_params, "n_trials": n, "log_lik": total_ll,
                "AIC": 2*n_params - 2*total_ll,
                "BIC": n_params*np.log(n) - 2*total_ll}


    # ═══════════════════════════════════════════════════════════
    # Consolidated CSV output (same convention as the lapse fitting cell)
    # ═══════════════════════════════════════════════════════════
    def save_fit_row(csv_path, pid, n_levels_requested, n_levels_actual, mle, ic,
                    extra=None):
        raw = mle.optimized_params_pd.iloc[0]
        keep_cols = [c for c in raw.index if not c.startswith("log_lik")]
        row = raw[keep_cols].to_dict()

        row = {
            "pid": pid,
            "n_levels_requested": n_levels_requested,
            "n_levels_actual": n_levels_actual,
            "n_params": ic["n_params"],
            "n_trials": ic["n_trials"],
            "log_lik_total": ic["log_lik"],
            "AIC": ic["AIC"],
            "BIC": ic["BIC"],
            "fit_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            **(extra or {}),
            **row,
        }
        new_row = pd.DataFrame([row])

        if os.path.exists(csv_path):
            existing = pd.read_csv(csv_path)
            mask_same = (existing["pid"] == pid) & (existing["n_levels_actual"] == n_levels_actual)
            existing = existing[~mask_same]
            combined = pd.concat([existing, new_row], ignore_index=True, sort=False)
        else:
            combined = new_row

        combined = combined.sort_values(["n_levels_actual", "pid"]).reset_index(drop=True)
        combined.to_csv(csv_path, index=False)
        return combined


    # ═══════════════════════════════════════════════════════════
    # Main
    # ═══════════════════════════════════════════════════════════
    def main():
        df, actual_levels = load_data()
        df = trim_extremes(df)

        # Parameter counts (no p_lapse in the no-lapse models):
        #   DDM:  a[2] + v_base[levels] + sv + sz + t0        = 2 + levels + 3
        #   SRDM: c[2] + d_base[levels] + B + r + t0          = 2 + levels + 3
        ddm_n_params  = 2 + actual_levels + 3
        srdm_n_params = 2 + actual_levels + 3

        print(f"\nDDM params: {ddm_n_params} ({actual_levels} drift rates, no lapse)")
        print(f"SRDM params: {srdm_n_params} ({actual_levels} d' values, no lapse)")

        print("\nCompiling models...")
        ddm_model = CmdStanModel(stan_file=STAN_DDM)
        srdm_model = CmdStanModel(stan_file=STAN_SRDM)

        for pid in PARTICIPANTS:
            print(f"\n{'='*60}")
            print(f"  {pid}  (N_LEVELS={actual_levels}, no lapse, trimmed)")
            print(f"{'='*60}")

            data = build_data(df, pid, actual_levels)
            print(f"  N_correct={data['N_correct']}  N_false={data['N_false']}  "
                f"t0_hi (min trimmed RT)={data['t0_hi']:.4f}")

            # DDM
            print(f"\n  --- DDM (no lapse) ---")
            ddm_mle = fit_ddm(ddm_model, data)
            ic = aic_bic(ddm_mle, ddm_n_params)
            save_fit_row(DDM_OUT, pid, N_LEVELS, actual_levels, ddm_mle, ic,
                        extra={"t0_hi": data["t0_hi"]})
            row = ddm_mle.optimized_params_pd.iloc[0]
            print(f"  a=[{row['a[1]']:.3f}, {row['a[2]']:.3f}]  t0={row['t0']:.4f}  "
                f"sv={row['sv']:.4f}  sz={row['sz']:.4f}")
            print(f"  LL={ic['log_lik']:.1f}  AIC={ic['AIC']:.1f}  BIC={ic['BIC']:.1f}")

            # SRDM
            print(f"\n  --- SRDM (no lapse) ---")
            srdm_mle = fit_srdm(srdm_model, data)
            ic2 = aic_bic(srdm_mle, srdm_n_params)
            save_fit_row(SRDM_OUT, pid, N_LEVELS, actual_levels, srdm_mle, ic2,
                        extra={"t0_hi": data["t0_hi"]})
            row = srdm_mle.optimized_params_pd.iloc[0]
            print(f"  c=[{row['c[1]']:.3f}, {row['c[2]']:.3f}]  B={row['B']:.3f}  "
                f"r={row['r']:.2f}  t0={row['t0']:.4f}")
            print(f"  LL={ic2['log_lik']:.1f}  AIC={ic2['AIC']:.1f}  BIC={ic2['BIC']:.1f}")

            delta = ic["AIC"] - ic2["AIC"]
            print(f"\n  ΔAIC(DDM-SRDM) = {delta:+.1f}  ({'DDM' if delta<0 else 'SRDM'} wins)")

        print(f"\nAll fits written/updated in:\n  {DDM_OUT}\n  {SRDM_OUT}")


    if __name__ == "__main__":
        main()


/var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/ipykernel_84233/1650709493.py:86: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trimmed = df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)
15:09:57 - cmdstanpy - INFO - Chain [1] start processing


N_LEVELS requested: 16, actual unique levels: 16
Trials before trimming: 22584
Trials after trimming (1%/99% per id x cell): 22060  (dropped 524, 2.3%)

DDM params: 21 (16 drift rates, no lapse)
SRDM params: 21 (16 d' values, no lapse)

Compiling models...

  jf  (N_LEVELS=16, no lapse, trimmed)
  N_correct=6264  N_false=894  t0_hi (min trimmed RT)=0.2080

  --- DDM (no lapse) ---
Chain [1] method = optimize
Chain [1] optimize
Chain [1] algorithm = lbfgs (Default)
Chain [1] lbfgs
Chain [1] init_alpha = 0.001 (Default)
Chain [1] tol_obj = 1e-12 (Default)
Chain [1] tol_rel_obj = 10000 (Default)
Chain [1] tol_grad = 1e-08 (Default)
Chain [1] tol_rel_grad = 1e+07 (Default)
Chain [1] tol_param = 1e-08 (Default)
Chain [1] history_size = 5 (Default)
Chain [1] jacobian = false (Default)
Chain [1] iter = 5000
Chain [1] save_iterations = false (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpabz0u_7q/vsuxlojy.json
Chain [1] 

15:10:39 - cmdstanpy - INFO - Chain [1] done processing


Chain [1] 170       2526.09   0.000485538     0.0823679           1           1      190
Chain [1] Optimization terminated normally:
Chain [1] Convergence detected: relative gradient magnitude is below tolerance
Chain [1] 
  a=[0.820, 2.292]  t0=0.2045  sv=0.5764  sz=0.1197
  LL=2549.8  AIC=-5057.6  BIC=-4913.2

  --- SRDM (no lapse) ---


15:10:39 - cmdstanpy - INFO - Chain [1] start processing


...

Chain [1] 99       4831.91    0.00061514      0.303189           1           1      116
Chain [1] Iter      log prob        ||dx||      ||grad||       alpha      alpha0  # evals  Notes
Chain [1] 100       4831.91   9.58469e-05      0.123714       0.935       0.935      117
Chain [1] Optimization terminated normally:
Chain [1] Convergence detected: relative gradient magnitude is below tolerance
Chain [1] 


15:12:16 - cmdstanpy - INFO - Chain [1] start processing


  a=[1.007, 1.909]  t0=0.2233  sv=0.7520  sz=0.1757
  LL=4857.0  AIC=-9672.0  BIC=-9525.6

  --- SRDM (no lapse) ---
Chain [1] method = optimize
Chain [1] optimize
Chain [1] algorithm = lbfgs (Default)
Chain [1] lbfgs
Chain [1] init_alpha = 0.001 (Default)
Chain [1] tol_obj = 1e-12 (Default)
Chain [1] tol_rel_obj = 10000 (Default)
Chain [1] tol_grad = 1e-08 (Default)
Chain [1] tol_rel_grad = 1e+07 (Default)
Chain [1] tol_param = 1e-08 (Default)
Chain [1] history_size = 5 (Default)
Chain [1] jacobian = false (Default)
Chain [1] iter = 5000
Chain [1] save_iterations = false (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpabz0u_7q/9xtiwas0.json
Chain [1] init = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpabz0u_7q/aqebss0u.json
Chain [1] random
Chain [1] seed = 71487
Chain [1] output
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpabz0u_7q/SRDM_rr98_c_only_nolapsenadf8_50/SRDM_rr98_c_only_

15:12:16 - cmdstanpy - INFO - Chain [1] done processing


Chain [1] 99       5353.28    0.00324221        4.7888           1           1      122
Chain [1] Iter      log prob        ||dx||      ||grad||       alpha      alpha0  # evals  Notes
Chain [1] 127       5353.29   0.000598459      0.451191           1           1      158
Chain [1] Optimization terminated normally:
Chain [1] Convergence detected: relative gradient magnitude is below tolerance
Chain [1] 
  c=[-0.184, 0.810]  B=2.392  r=13.80  t0=0.1437
  LL=5373.7  AIC=-10705.4  BIC=-10559.0

  ΔAIC(DDM-SRDM) = +1033.4  (SRDM wins)

All fits written/updated in:
  fits_ddm_nolapse.csv
  fits_srdm_nolapse.csv


In [11]:
fit_ddm = pd.read_csv("fits_ddm_nolapse.csv")
print(fit_ddm["sv"])
fit_ddm

0    0.025077
1    0.028234
2    0.030122
3    0.001219
4    0.001284
5    0.002117
6    0.001299
7    0.001645
8    0.003228
Name: sv, dtype: float64


,pid,n_levels_requested,n_levels_actual,n_params,n_trials,log_lik_total,AIC,BIC,fit_time_utc,t0_hi,...,v_full[55],v_full[56],v_full[57],v_full[58],v_full[59],v_full[60],v_full[61],v_full[62],v_full[63],v_full[64]
0,jf,9,9,14,7167,2497.757333,-4967.514666,-4871.233272,2026-07-22T13:09:52+00:00,0.208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,kr,9,9,14,7029,2901.590971,-5775.181942,-5679.172746,2026-07-22T13:10:08+00:00,0.206,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,nh,9,9,14,7892,4795.104229,-9562.208459,-9464.577991,2026-07-22T13:10:27+00:00,0.226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,jf,16,16,21,7158,2533.414279,-5024.828559,-4880.432855,2026-07-22T14:44:01+00:00,0.208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,kr,16,16,21,7023,2977.948047,-5913.896094,-5769.900234,2026-07-22T14:44:38+00:00,0.206,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,nh,16,16,21,7879,4818.372764,-9594.745527,-9448.334445,2026-07-22T14:45:04+00:00,0.226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,jf,33,32,37,7114,2550.252313,-5026.504626,-4772.321288,2026-07-22T14:48:42+00:00,0.207,...,2.313578,2.312584,2.446912,2.627625,2.340674,2.512734,2.682303,2.556975,2.533932,2.829213
7,kr,33,32,37,6986,2960.558934,-5847.117869,-5593.606322,2026-07-22T14:49:21+00:00,0.205,...,2.887160,3.276991,3.659209,3.739175,3.887260,4.126948,3.923744,4.385333,4.246013,4.482476
8,nh,33,32,37,7852,4824.918383,-9575.836766,-9318.001394,2026-07-22T14:49:36+00:00,0.226,...,3.242827,3.512932,3.618415,3.142004,3.498816,3.737790,4.121931,3.610252,3.882234,3.594753
